In [ ]:
EXECUTE_EXTRACTION = True

MIN_IMAGE_SIZE = 100
MAX_IMAGE_SIZE = 4096
MIN_AREA = 10000
QUALITY = 95
SKIP_EXISTING = True

print(f"Extraction mode: {'ACTIVE' if EXECUTE_EXTRACTION else 'DRY RUN'}")


In [ ]:
import sys
from pathlib import Path
from datetime import datetime
import json

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_PDFS_DIR = PROJECT_ROOT / "data" / "raw_pdfs"
OUTPUT_IMAGES_DIR = PROJECT_ROOT / "data" / "academic_dataset" / "images"
METADATA_DIR = PROJECT_ROOT / "data" / "academic_dataset" / "metadata"
PROGRESS_FILE = PROJECT_ROOT / "data" / "extraction_progress.json"

OUTPUT_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"PDF source: {RAW_PDFS_DIR}")
print(f"Image output: {OUTPUT_IMAGES_DIR}")


In [ ]:
def get_extraction_stats():
    """Get statistics about current extraction status."""
    pdf_files = list(RAW_PDFS_DIR.glob("*.pdf")) if RAW_PDFS_DIR.exists() else []
    image_files = list(OUTPUT_IMAGES_DIR.glob("*.png")) + list(OUTPUT_IMAGES_DIR.glob("*.jpg"))
    
    unique_pdfs = set()
    for img in image_files:
        parts = img.stem.split("_page_")
        if parts:
            unique_pdfs.add(parts[0])
    
    return {
        "total_pdfs": len(pdf_files),
        "total_images": len(image_files),
        "processed_pdfs": len(unique_pdfs),
        "avg_images_per_pdf": round(len(image_files) / len(unique_pdfs), 2) if unique_pdfs else 0,
        "pending_pdfs": len(pdf_files) - len(unique_pdfs),
    }

stats = get_extraction_stats()
print("=" * 50)
print("EXTRACTION STATISTICS")
print("=" * 50)
print(f"Total PDFs:          {stats['total_pdfs']:,}")
print(f"Processed PDFs:      {stats['processed_pdfs']:,}")
print(f"Pending PDFs:        {stats['pending_pdfs']:,}")
print(f"Total Images:        {stats['total_images']:,}")
print(f"Avg Images/PDF:      {stats['avg_images_per_pdf']}")
print("=" * 50)


In [ ]:
import io
import hashlib
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

try:
    import fitz
except ImportError:
    print("Installing PyMuPDF...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pymupdf", "-q"])
    import fitz

from PIL import Image
import numpy as np


@dataclass
class ExtractedImage:
    """Represents an extracted image."""
    image_id: str
    source_pdf: str
    page_number: int
    image_index: int
    width: int
    height: int
    format: str
    output_path: Optional[Path] = None
    caption: Optional[str] = None
    bbox: Optional[Tuple[float, float, float, float]] = None


class PDFMiner:
    """
    Extract images from PDF files.
    
    Uses PyMuPDF (fitz) for extraction with quality filtering.
    """
    
    def __init__(
        self,
        output_dir: Path,
        min_size: int = 100,
        max_size: int = 4096,
        min_area: int = 10000,
    ):
        self.output_dir = Path(output_dir)
        self.min_size = min_size
        self.max_size = max_size
        self.min_area = min_area
        self.output_dir.mkdir(parents=True, exist_ok=True)
    
    def extract_from_pdf(
        self,
        pdf_path: Path,
        skip_existing: bool = True,
    ) -> List[ExtractedImage]:
        """
        Extract all valid images from a PDF.
        
        Args:
            pdf_path: Path to PDF file
            skip_existing: Skip if images already extracted
            
        Returns:
            List of ExtractedImage objects
        """
        pdf_id = pdf_path.stem
        extracted = []
        
        try:
            doc = fitz.open(pdf_path)
        except Exception as e:
            print(f"Failed to open PDF | pdf={pdf_id} | error={e}")
            return []
        
        try:
            for page_num in range(len(doc)):
                page = doc[page_num]
                image_list = page.get_images(full=True)
                
                for img_idx, img_info in enumerate(image_list):
                    xref = img_info[0]
                    
                    image_id = f"{pdf_id}_page_{page_num + 1}_img_{img_idx + 1}"
                    output_path = self.output_dir / f"{image_id}.png"
                    
                    if skip_existing and output_path.exists():
                        continue
                    
                    try:
                        base_image = doc.extract_image(xref)
                        image_bytes = base_image["image"]
                        
                        pil_image = Image.open(io.BytesIO(image_bytes))
                        width, height = pil_image.size
                        
                        if not self._is_valid_image(width, height):
                            continue
                        
                        if pil_image.mode not in ["RGB", "L"]:
                            pil_image = pil_image.convert("RGB")
                        
                        pil_image.save(output_path, "PNG")
                        
                        extracted.append(ExtractedImage(
                            image_id=image_id,
                            source_pdf=pdf_id,
                            page_number=page_num + 1,
                            image_index=img_idx + 1,
                            width=width,
                            height=height,
                            format="PNG",
                            output_path=output_path,
                        ))
                        
                    except Exception as e:
                        continue
                        
        finally:
            doc.close()
        
        return extracted
    
    def _is_valid_image(self, width: int, height: int) -> bool:
        """Check if image meets quality criteria."""
        if width < self.min_size or height < self.min_size:
            return False
        if width > self.max_size or height > self.max_size:
            return False
        
        if width * height < self.min_area:
            return False
        
        aspect = max(width, height) / min(width, height)
        if aspect > 10:
            return False
        
        return True


print("PDFMiner class defined.")


In [ ]:
def preview_extraction(pdf_dir: Path, num_samples: int = 3):
    """Preview extraction from a few PDFs without saving."""
    pdf_files = list(pdf_dir.glob("*.pdf"))[:num_samples]
    
    if not pdf_files:
        print("No PDF files found!")
        return
    
    print(f"Previewing extraction from {len(pdf_files)} PDFs...\n")
    
    for pdf in pdf_files:
        print(f"PDF: {pdf.name}")
        
        try:
            doc = fitz.open(pdf)
            total_images = 0
            valid_images = 0
            
            for page_num in range(min(len(doc), 10)):
                page = doc[page_num]
                images = page.get_images(full=True)
                total_images += len(images)
                
                for img_info in images:
                    try:
                        xref = img_info[0]
                        base_image = doc.extract_image(xref)
                        pil_image = Image.open(io.BytesIO(base_image["image"]))
                        w, h = pil_image.size
                        
                        if w >= MIN_IMAGE_SIZE and h >= MIN_IMAGE_SIZE and w * h >= MIN_AREA:
                            valid_images += 1
                    except:
                        pass
            
            doc.close()
            
            print(f"  Pages: {len(doc)} | Raw images: {total_images} | Valid: {valid_images}")
            
        except Exception as e:
            print(f"  Error: {e}")
        
        print()

preview_extraction(RAW_PDFS_DIR, num_samples=3)


In [ ]:
from typing import Dict, Any
import time


def load_extraction_progress() -> Dict[str, Any]:
    """Load extraction progress."""
    if PROGRESS_FILE.exists():
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {"processed": [], "total_images": 0, "errors": []}


def save_extraction_progress(progress: Dict[str, Any]):
    """Save extraction progress."""
    with open(PROGRESS_FILE, "w") as f:
        json.dump(progress, f, indent=2)


def batch_extract(
    pdf_dir: Path,
    output_dir: Path,
    resume: bool = True,
    dry_run: bool = True,
    max_pdfs: int = None,
) -> Dict[str, Any]:
    """
    Extract images from all PDFs with progress tracking.
    
    Args:
        pdf_dir: Directory containing PDFs
        output_dir: Directory for extracted images
        resume: Resume from checkpoint
        dry_run: Don't actually extract
        max_pdfs: Maximum PDFs to process (None = all)
        
    Returns:
        Statistics dictionary
    """
    miner = PDFMiner(
        output_dir=output_dir,
        min_size=MIN_IMAGE_SIZE,
        max_size=MAX_IMAGE_SIZE,
        min_area=MIN_AREA,
    )
    
    if resume:
        progress = load_extraction_progress()
    else:
        progress = {"processed": [], "total_images": 0, "errors": []}
    
    processed_set = set(progress["processed"])
    
    pdf_files = sorted(pdf_dir.glob("*.pdf"))
    if max_pdfs:
        pdf_files = pdf_files[:max_pdfs]
    
    pending = [p for p in pdf_files if p.stem not in processed_set]
    
    print("=" * 60)
    print(f"BATCH EXTRACTION | mode={'DRY RUN' if dry_run else 'ACTIVE'}")
    print(f"Total PDFs: {len(pdf_files)} | Pending: {len(pending)}")
    print("=" * 60)
    
    if dry_run:
        print("\n[DRY RUN] Would extract from pending PDFs...")
        print("Set EXECUTE_EXTRACTION = True to actually extract.")
        return progress
    
    start_time = time.time()
    batch_images = 0
    
    for idx, pdf_path in enumerate(pending, 1):
        try:
            images = miner.extract_from_pdf(pdf_path, skip_existing=SKIP_EXISTING)
            
            progress["processed"].append(pdf_path.stem)
            progress["total_images"] += len(images)
            batch_images += len(images)
            
            if idx % 10 == 0:
                elapsed = time.time() - start_time
                rate = idx / elapsed if elapsed > 0 else 0
                eta = (len(pending) - idx) / rate if rate > 0 else 0
                
                print(f"  [{idx}/{len(pending)}] Images extracted: {batch_images} | "
                      f"Rate: {rate:.1f} PDF/s | ETA: {eta/60:.1f}min")
                
                save_extraction_progress(progress)
                
        except KeyboardInterrupt:
            print("\n[INTERRUPTED] Saving progress...")
            save_extraction_progress(progress)
            break
            
        except Exception as e:
            progress["errors"].append({"pdf": pdf_path.stem, "error": str(e)})
    
    save_extraction_progress(progress)
    
    elapsed = time.time() - start_time
    print("\n" + "=" * 60)
    print("EXTRACTION COMPLETE")
    print(f"  PDFs processed: {idx}")
    print(f"  Images extracted: {batch_images}")
    print(f"  Time: {elapsed/60:.1f} minutes")
    print(f"  Errors: {len(progress['errors'])}")
    print("=" * 60)
    
    return progress


print("batch_extract() function defined.")


In [ ]:
if EXECUTE_EXTRACTION:
    print("Starting batch extraction...")
    print("Press Ctrl+C to interrupt (progress will be saved).")
    print()
    
    result = batch_extract(
        pdf_dir=RAW_PDFS_DIR,
        output_dir=OUTPUT_IMAGES_DIR,
        resume=True,
        dry_run=False,
    )
else:
    print("[SKIPPED] Extraction not executed.")
    print("Set EXECUTE_EXTRACTION = True at the top to enable extraction.")
    print("\nTo extract manually, run:")
    print("  batch_extract(RAW_PDFS_DIR, OUTPUT_IMAGES_DIR, dry_run=False)")


In [ ]:
import random

def analyze_images(image_dir: Path, sample_size: int = 100):
    """Analyze quality of extracted images."""
    image_files = list(image_dir.glob("*.png")) + list(image_dir.glob("*.jpg"))
    
    if not image_files:
        print("No images found!")
        return
    
    samples = random.sample(image_files, min(sample_size, len(image_files)))
    
    widths = []
    heights = []
    sizes_kb = []
    
    for img_path in samples:
        try:
            img = Image.open(img_path)
            widths.append(img.size[0])
            heights.append(img.size[1])
            sizes_kb.append(img_path.stat().st_size / 1024)
            img.close()
        except:
            pass
    
    print("=" * 50)
    print("IMAGE QUALITY ANALYSIS")
    print(f"(Based on {len(widths)} samples)")
    print("=" * 50)
    print(f"Width:   min={min(widths)}, max={max(widths)}, avg={np.mean(widths):.0f}")
    print(f"Height:  min={min(heights)}, max={max(heights)}, avg={np.mean(heights):.0f}")
    print(f"Size:    min={min(sizes_kb):.0f}KB, max={max(sizes_kb):.0f}KB, avg={np.mean(sizes_kb):.0f}KB")
    print("=" * 50)
    
    small = sum(1 for w, h in zip(widths, heights) if w < 300 or h < 300)
    medium = sum(1 for w, h in zip(widths, heights) if 300 <= w < 800 and 300 <= h < 800)
    large = sum(1 for w, h in zip(widths, heights) if w >= 800 or h >= 800)
    
    print("\nSize Distribution:")
    print(f"  Small (<300px):    {small} ({small/len(widths)*100:.1f}%)")
    print(f"  Medium (300-800):  {medium} ({medium/len(widths)*100:.1f}%)")
    print(f"  Large (>800px):    {large} ({large/len(widths)*100:.1f}%)")

analyze_images(OUTPUT_IMAGES_DIR)


In [ ]:
import matplotlib.pyplot as plt

def show_sample_images(image_dir: Path, num_samples: int = 6):
    """Display random sample of extracted images."""
    image_files = list(image_dir.glob("*.png")) + list(image_dir.glob("*.jpg"))
    
    if not image_files:
        print("No images found!")
        return
    
    samples = random.sample(image_files, min(num_samples, len(image_files)))
    
    cols = 3
    rows = (num_samples + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
    axes = axes.flatten() if num_samples > 1 else [axes]
    
    for ax, img_path in zip(axes, samples):
        try:
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(img_path.stem[:30] + "...", fontsize=8)
            ax.axis("off")
        except Exception as e:
            ax.text(0.5, 0.5, f"Error: {e}", ha="center", va="center")
            ax.axis("off")
    
    for ax in axes[len(samples):]:
        ax.axis("off")
    
    plt.tight_layout()
    plt.show()

show_sample_images(OUTPUT_IMAGES_DIR, num_samples=6)


In [ ]:
def generate_extraction_report():
    """Generate comprehensive extraction report."""
    stats = get_extraction_stats()
    progress = load_extraction_progress()
    
    image_files = list(OUTPUT_IMAGES_DIR.glob("*.png")) + list(OUTPUT_IMAGES_DIR.glob("*.jpg"))
    total_size_mb = sum(f.stat().st_size for f in image_files) / (1024 * 1024)
    
    report = f"""
================================================================================
                       IMAGE EXTRACTION REPORT
                       {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
================================================================================

SOURCE DATA
-----------
Total PDFs:           {stats['total_pdfs']:,}
Processed PDFs:       {stats['processed_pdfs']:,}
Pending PDFs:         {stats['pending_pdfs']:,}

EXTRACTED IMAGES
----------------
Total Images:         {stats['total_images']:,}
Total Size:           {total_size_mb:,.1f} MB
Avg Images/PDF:       {stats['avg_images_per_pdf']}
Avg Image Size:       {total_size_mb * 1024 / stats['total_images']:.1f} KB (per image)

QUALITY FILTERS
---------------
Min Size:             {MIN_IMAGE_SIZE}px
Max Size:             {MAX_IMAGE_SIZE}px
Min Area:             {MIN_AREA}px

ERRORS
------
Failed PDFs:          {len(progress.get('errors', []))}

    
    print(report)
    
    report_path = METADATA_DIR / f"extraction_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    report_path.write_text(report)
    print(f"Report saved to: {report_path}")


generate_extraction_report()
